<a href="https://colab.research.google.com/github/nrios256-create/analisis_datos/blob/main/Salud_mental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Salud_Mental_20260410.csv")

# Visualizamos las primeras filas
df.head()

,diagnostico_ingreso,codigo_dx_ingreso,Menor a 1,De 1 a 4,De 5 a 9,De 10 a 14,De 15 a 19,De 20 a 49,De 50 a 64,65 Y MAS,Total,Año diagnóstico
0,TRASTORNO MIXTO DE ANSIEDAD Y DEPRESION,F412,0,0,0,10,9,31,8,2,60,2023
1,EPISODIO DEPRESIVO GRAVE SIN SINTOMAS PSICOTICOS,F322,0,0,0,10,10,31,4,3,58,2023
2,EPISODIO DEPRESIVO MODERADO,F321,0,0,0,3,2,11,1,4,21,2023
3,"TRASTORNO AFECTIVO BIPOLAR, NO ESPECIFICADO",F319,0,0,0,1,1,12,4,1,19,2023
4,"ESQUIZOFRENIA, NO ESPECIFICADA",F209,0,0,0,1,0,14,0,0,15,2023


In [3]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 142 entries, 0 to 141
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   diagnostico_ingreso  142 non-null    object
 1   codigo_dx_ingreso    142 non-null    object
 2   Menor a 1            142 non-null    int64 
 3   De 1 a 4             142 non-null    int64 
 4   De 5 a 9             142 non-null    int64 
 5   De 10 a 14           142 non-null    int64 
 6   De 15 a 19           142 non-null    int64 
 7   De 20 a 49           142 non-null    int64 
 8   De 50 a 64           142 non-null    int64 
 9   65 Y MAS             142 non-null    int64 
 10  Total                142 non-null    int64 
 11  Año diagnóstico      142 non-null    int64 
dtypes: int64(10), object(2)
memory usage: 13.4+ KB


(142, 12)

In [4]:
df.columns

Index(['diagnostico_ingreso', 'codigo_dx_ingreso', 'Menor a 1', 'De 1 a 4',
       'De 5 a 9', 'De 10 a 14', 'De 15 a 19', 'De 20 a 49', 'De 50 a 64',
       '65 Y MAS', 'Total', 'Año diagnóstico'],
      dtype='object')

In [20]:
# Aquí utilicé la función melt porque necesitaba transformar el dataset,
# ya que las edades estaban como columnas y así no se puede analizar fácilmente por grupos

df_largo = df.melt(

    # Aquí utilicé id_vars porque esta opción me permite mantener fijas las columnas
    # que identifican cada registro, en este caso el diagnóstico y el código,
    # y me ayudó a no perder esa información al transformar los datos
    id_vars=["diagnostico_ingreso", "codigo_dx_ingreso"],

    # Aquí utilicé var_name para crear una nueva columna donde se guardan
    # los nombres de los grupos de edad, lo que me ayudó a organizar mejor la información
    var_name="grupo_edad",

    # Aquí utilicé value_name para guardar los valores que estaban en las columnas,
    # es decir, la cantidad de casos, lo que me permitió trabajar con esos datos de forma más clara
    value_name="conteo"
)

# Aquí utilicé head() para visualizar las primeras filas del dataset transformado
# y así verificar que el proceso de melt se hizo correctamente
df_largo.head()

,diagnostico_ingreso,codigo_dx_ingreso,grupo_edad,conteo
0,TRASTORNO MIXTO DE ANSIEDAD Y DEPRESION,F412,Menor a 1,0
1,EPISODIO DEPRESIVO GRAVE SIN SINTOMAS PSICOTICOS,F322,Menor a 1,0
2,EPISODIO DEPRESIVO MODERADO,F321,Menor a 1,0
3,"TRASTORNO AFECTIVO BIPOLAR, NO ESPECIFICADO",F319,Menor a 1,0
4,"ESQUIZOFRENIA, NO ESPECIFICADA",F209,Menor a 1,0


In [21]:
# Aquí lo que hice fue filtrar el dataset para eliminar columnas que no me sirven
# para el análisis, como "Total" y "Año diagnóstico", porque no representan grupos de edad

df_largo = df_largo[

    # Aquí utilicé ~ (negación) porque quiero quedarme con todo lo que NO sea
    # "Total" ni "Año diagnóstico"
    ~df_largo["grupo_edad"].isin(["Total", "Año diagnóstico"])

    # Aquí utilicé isin() porque me permite identificar si los valores de la columna
    # "grupo_edad" coinciden con esos nombres específicos, lo que me ayudó a filtrarlos fácilmente
]

In [22]:
# Aquí utilicé un diccionario porque necesitaba cambiar los nombres originales
# de los grupos de edad por otros más claros y más fáciles de interpretar
diccionario_edades = {
    "Menor a 1": "Primera infancia",
    "De 1 a 4": "Infancia temprana",
    "De 5 a 9": "Niñez",
    "De 10 a 14": "Pre-adolescencia",
    "De 15 a 19": "Adolescencia",
    "De 20 a 49": "Adulto joven",
    "De 50 a 64": "Adulto",
    "65 Y MAS": "Adulto mayor"
}

df_largo["grupo_edad"] = df_largo["grupo_edad"].map(diccionario_edades)
# Aquí utilicé map() porque esta función me permite reemplazar
# los valores originales de la columna por los nuevos valores del diccionario

In [12]:
df_largo = df_largo[df_largo["conteo"] > 0] #Limpiar datos (quitar ceros)

In [23]:
# Aquí utilicé groupby() porque necesitaba agrupar los datos
# por grupo de edad y por diagnóstico, para poder ver cuántos casos había en cada combinación

tabla = df_largo.groupby(
    ["grupo_edad", "diagnostico_ingreso"]
)["conteo"].sum().reset_index()

# Aquí seleccioné la columna conteo porque esa es la que contiene la cantidad de casos

# Luego utilicé sum() porque necesitaba sumar los valores de conteo
# y así obtener el total de casos por cada enfermedad en cada grupo de edad

# Finalmente utilicé reset_index() porque después de agrupar los datos
# el resultado queda con índices, y con esta función lo convierto otra vez en un DataFrame normal
# para poder seguir trabajando más fácil con la información

In [24]:
# Aquí lo que hice fue calcular la proporción de cada enfermedad dentro de cada grupo de edad

tabla["proporcion"] = tabla.groupby("grupo_edad")["conteo"].transform(

    # Aquí utilicé una función lambda porque necesitaba dividir cada valor
    # entre el total de su grupo de edad

    lambda x: x / x.sum()
)

# Aquí utilicé groupby() nuevamente para trabajar dentro de cada grupo de edad,
# es decir, calcular proporciones de forma independiente en cada grupo

# Utilicé transform() porque me permite hacer el cálculo sin perder la estructura del DataFrame,
# es decir, mantiene el mismo tamaño de la tabla original

# Esto me ayudó a obtener la proporción de cada enfermedad
# y así poder identificar cuál es la más representativa en cada grupo de edad

In [27]:
# Aquí lo que hice fue encontrar la enfermedad con mayor proporción en cada grupo de edad

resultado = tabla.loc[

    # Aquí utilicé groupby() para trabajar por cada grupo de edad
    # y buscar dentro de cada uno cuál es la proporción más alta

    tabla.groupby("grupo_edad")["proporcion"].idxmax()
]

# Aquí utilicé idxmax() porque esta función me devuelve la posición
# donde está el valor máximo de la proporción en cada grupo

# Luego utilicé loc[] porque con esas posiciones puedo traer
# las filas completas del DataFrame, es decir, la enfermedad correspondiente

# Esto me ayudó a identificar directamente cuál es la enfermedad
# más representativa en cada grupo de edad
resultado

,grupo_edad,diagnostico_ingreso,conteo,proporcion
12,Adolescencia,EPISODIO DEPRESIVO GRAVE SIN SINTOMAS PSICOTICOS,33,0.333333
169,Adulto,TRASTORNO MIXTO DE ANSIEDAD Y DEPRESION,13,0.175676
210,Adulto joven,EPISODIO DEPRESIVO GRAVE SIN SINTOMAS PSICOTICOS,54,0.157434
301,Adulto mayor,"DELIRIO, NO ESPECIFICADO",23,0.261364
397,Infancia temprana,CONVULSIONES DISOCIATIVAS,1,1.000000
507,Niñez,EPISODIO DEPRESIVO GRAVE SIN SINTOMAS PSICOTICOS,3,0.375000
606,Pre-adolescencia,EPISODIO DEPRESIVO GRAVE SIN SINTOMAS PSICOTICOS,31,0.333333
694,Primera infancia,CONVULSIONES DISOCIATIVAS,1,1.000000


#INTERPRETACION

Bueno, a partir del análisis de los datos de salud mental del Hospital San Vicente de Paul, lo que hice fue identificar cuál es la enfermedad que más se repite en cada grupo de edad.

Se puede ver que en varios grupos como adolescencia, niñez, pre-adolescencia y adulto joven, la enfermedad que más aparece es el episodio depresivo grave sin síntomas psicóticos. Esto básicamente muestra que los problemas relacionados con depresión están muy presentes en esas edades.

En el caso de los adultos, la enfermedad que más predomina es el trastorno mixto de ansiedad y depresión, lo que indica que en esas edades ya no es solo depresión, sino que también hay ansiedad combinada.

Para los adultos mayores, lo que más aparece es el delirio no especificado, que probablemente está relacionado con temas de deterioro cognitivo o enfermedades de la edad.

También se observa que en los grupos más pequeños, como primera infancia e infancia temprana, aparecen las convulsiones disociativas como la más frecuente, aunque ahí los datos son muy pocos, entonces no se puede generalizar tanto.

En general, se puede decir que la depresión es una de las enfermedades más representativas en la mayoría de los grupos de edad, especialmente en personas jóvenes y adultas.